# Objective
This notebook compares the [BOOST](https://arxiv.org/pdf/2404.08826) scheduling policy with [PCS](https://www.usenix.org/conference/osdi24/presentation/bin-faisal).
The objective is two fold:
1. Understand whether BOOST can provide similar predictability in job completion times as PCS
2. Whether minimizing prediction error is the same as minimzing tail job completion times

## Main results
Summary of main results

## Concluding statement

# Background

## Predictability vs. Performance
In order to provide accurate job completion time predictions, we need to limit the degree of preemption experienced by jobs. By controlling the degree of preemption - from no preemption as in FIFO to *unbounded* preemption as in Shortest-Job-First (SJF) - we can achieve different trade-offs between predictability (low prediction error) and performance (low Job Completion Times (JCTs)).

The main question is what should the underlying scheduling policy be? The PCS paper argues that it should be Weighted-Fair-Queues (WFQs).  

## PCS
The main insight in PCS is that different WFQ configurations span the space of scheduling options and hence different objectives: from well-known points such as FIFO (WFQ with a single queue) and SJF (queues with small jobs assigned exponentially larger weights) to intermediate scheduling points.
PCS searches different WFQ configurations (e.g., number of queues, queue weights etc.) to achieve varying trade-offs between competing objectives.

## BOOST
BOOST is designed to provide tail-optimal JCTs for light-tailed workloads (e.g., uniform job-sizes) while improving upon FIFO (which has traditionally been known to be tail-optimal).
BOOST works similar to FIFO but in order to achieve its objective, it pretendes small jobs arrive earlier than their true arrival times; referred to as the *boosted arrival time*.
Thus jobs with earlier (smaller) boosted arrival times are served first.
The boosted arrival time of a job is:
$$
\text{boosted arrival time} = \text{arrival time} - b(s)
$$
Where $s$ is a job's size and $b(s)$ is the boost function.
The specific boost function the paper uses is:
$$
b(s) = \frac{1}{\gamma} \log \left( \frac{1}{1 - \exp(-\gamma \cdot s)} \right)
$$
Intuitively, smaller the job, larger the value of the boost function and hence smaller the boosted arrival time.
The scheme is parameterized by $\gamma$, which is a hyperparameter allowing BOOST to approximate extreme points like FIFO (larger $\gamma$) and SJF (smaller $\gamma$) as well as intermediate points.

# Experiment setting
The experiments use simulations to evaluate the ability of both scheduling policies to achieve Pareto-optimal trade-offs between the specified objectives.

For a given workload, PCS aims to find Pareto-optimal WFQ configurations via a simulation-based search framework (details in the paper).
The same framework can also be used to find pareto-optimal values for $\gamma$ for the BOOST policy.


For experiment 2, the specified objectives are minimizing 1. average JCT and 2. p99 JCT.

## Workloads and Setup
For the evaluation, two workloads from the PCS paper (see Workload 1 and 3, Table 1) are considered.
Workload 1 is light-tailed while workload 3 is heavy-tailed.
Exact job-size distributions are given at the end. 
For simplicity, the simulations are for 1-GPU running with the system load set to be 80%.

## Metrics
The evaluation considers average JCT vs. average prediction error.
For a single job, prediction error is defined as:
$$
100.0 * (\text{JCT} - \text{Predicted JCT}) / \text{Predicted JCT}
$$


# Expt 1: PCS vs. BOOST - avg JCT vs. avg Prediction Error
For experiment 1, the specified objectives to the search framework are: minimizing 1. average JCT and 2. average prediction error.
The objective is to see whether Pareto-optimal BOOST schedulers are better than Pareto-optimal WFQ configurations. 